In [1]:
!pip install oracledb sqlalchemy pandas

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import pandas as pd
#create_engine функція, яка створює об'єкт "двигуна"(Engine)
#engine слугує центральною точкою для підключення до БД
from sqlalchemy import create_engine

username = "system"
password = "1310958OracleDB"
host = "localhost"
port = "1521"
service_name = "FREE"

path = r"C:/Users/stepa/Desktop/АДІС/KP_ADIS_Macroeconomic_analysis/ADIS_Project_clean_datasets/"

#Підключення до Oracle
engine = create_engine(f"oracle+oracledb://{username}:{password}@{host}:{port}/{service_name}")

print("Підключення до Oracle успішне!")

def load_to_database(filename, table_name, rename_columns=None):
    print(f"\nЗавантажуємо {filename} в таблицю {table_name}...")

    #low_memory=False примушує pandas прочитати весь файл одразу, 
    # а не частинами, що дозволяє правильно визначити типи даних у всіх колонках (уникнення DtypeWarning)
    df = pd.read_csv(path + filename, low_memory = False)
    if rename_columns:
        df = df.rename(columns = rename_columns)

    #Для stg_assets додаємо колонку asset_name
    if table_name == "stg_assets":
        df["asset_name"] = df.get("brand_name").fillna(df.get("name")).fillna("Unknown")

    #Видаляємо колонки, яких немає в таблиці (load_id і load_timestamp генеруються автоматично)
    if table_name == "stg_assets":
        columns_to_keep = [
            "trade_date", "open_price", "high_price", "low_price", "close_price",
            "volume", "ticker", "asset_name", "asset_type", "country", 
            "industry_tag", "dividends", "stock_splits", "daily_return"
        ]
        df = df[columns_to_keep]

    #Замінюємо NaN на None (Oracle краще працює з NULL)
    df = df.where(pd.notnull(df), None)

    if "trade_date" in df.columns:
        df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")
        df["trade_date"] = df["trade_date"].dt.tz_localize(None)
        
    if "year_date" in df.columns:
        df["year_date"] = pd.to_datetime(df["year_date"], errors="coerce")
        df["year_date"] = df["year_date"].dt.tz_localize(None)

    #Завантажуємо в базу
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="append",  # додаємо дані, не видаляємо старі
        index=False,
        #method="multi",  # швидше завантаження
        chunksize=10000  # порціями по 10 тисяч рядків
    )
    
    print(f"Успішно завантажено {len(df):,} рядків у {table_name}")

load_to_stage(
    filename="assets_clean.csv",
    table_name="stg_assets",
    rename_columns={
        "date": "trade_date",
        "open": "open_price",
        "high": "high_price",
        "low": "low_price",
        "close": "close_price"
    }
)

load_to_stage(
    filename="gdp_clean.csv",
    table_name="stg_gdp",
    rename_columns={"year": "year_date"}
)

load_to_stage(
    filename="inflation_clean.csv",
    table_name="stg_inflation",
    rename_columns={"year": "year_date"}
)

load_to_stage(
    filename="unemployment_clean.csv",
    table_name="stg_unemployment",
    rename_columns={
        "year": "year_date",
        "unemployment_rate_value": "unemployment_rate"
    }
)

load_to_stage(
    filename="real_interest_clean.csv",
    table_name="stg_real_interest",
    rename_columns={"year": "year_date"}
)

print("\nВсі дані успіщшно завантажені в stage зону!")